# Домашнее задание 1

**Студент:** Почуев Эдуард Константинович

## Задание 1

Базовый счёт, банковские операции и история.

In [22]:
import csv
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Any


class Account:
    _account_counter = 1000
    account_type = "account"
    allowed_operations = ("deposit", "withdraw")

    def __init__(self, account_holder: str, balance: float = 0) -> None:
        self._validate_holder(account_holder)
        self._validate_number(balance)

        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder: str = account_holder
        self._balance: float = float(balance)
        self.operations_history: list[dict[str, Any]] = []

        self.account_number: str = f"ACC-{Account._account_counter:04d}"
        Account._account_counter += 1

    @staticmethod
    def _validate_holder(account_holder: str) -> None:
        if not isinstance(account_holder, str):
            raise TypeError("Имя владельца должно быть строкой")

        name_pattern = r"[A-ZА-ЯЁ][A-Za-zА-Яа-яЁё]*"

        if not re.fullmatch(f"{name_pattern} {name_pattern}", account_holder):
            raise ValueError("Укажите имя и фамилию с заглавных букв")

    @staticmethod
    def _validate_number(value: float) -> None:
        if not isinstance(value, (int, float)):
            raise TypeError("Значение должно быть числом")

    @staticmethod
    def _validate_amount(amount: float) -> None:
        Account._validate_number(amount)
        if amount <= 0:
            raise ValueError("Сумма операции должна быть положительной")

    def _add_operation(self, operation_type: str, amount: float, status: str) -> None:
        """Сохраняет историческую информацию об операции"""
        self.operations_history.append({
            "type": operation_type,
            "amount": amount,
            "datetime": datetime.now(),
            "balance": self._balance,
            "status": status,
        })

    def deposit(self, amount: float) -> bool:
        self._validate_amount(amount)
        self._balance += amount
        self._add_operation("deposit", amount, "success")

        return True

    def withdraw(self, amount: float) -> bool:
        self._validate_amount(amount)
        return self._withdraw(amount)

    def _withdraw(self, amount: float) -> bool:
        """
        Выполняет снятие средств для уже проверенной суммы. Возвращает True, если средств достаточно и операция выполнена,
        и False - иначе
        """
        if amount > self._balance:
            self._add_operation("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._add_operation("withdraw", amount, "success")

        return True

    def get_balance(self) -> float:
        return self._balance

    def get_history(self) -> list[dict[str, Any]]:
        return self.operations_history

    def show_large_operations(self, n: int, min_amount: float) -> list[dict[str, Any]]:
        """Выводит последние n успешных операций от min_amount"""
        large_operations = [
            operation for operation in self.operations_history
            if operation["status"] == "success"
            and operation["amount"] >= min_amount
        ]
        large_operations.sort(key=lambda item: item["datetime"], reverse=True)
        result = large_operations[:n]

        for operation in result:
            print(
                f"{operation['datetime']:%d.%m.%Y %H:%M:%S} | "
                f"{operation['type']} | сумма: {operation['amount']:.2f} | "
                f"баланс: {operation['balance']:.2f}"
            )

        return result

    def clean_history(self, rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
        """Очищает загруженные транзакции; возвращает список валидных операций, отсортированных по дате"""
        operations: list[dict[str, Any]] = []

        for row in rows:
            if row.get("account_type") != self.account_type:
                continue

            operation_type = row.get("operation")
            status = row.get("status")

            if operation_type not in self.allowed_operations:
                continue
            if status not in ("success", "fail"):
                continue

            try:
                amount = float(row.get("amount"))
                balance_after = float(row.get("balance_after"))

                if amount < 0 or balance_after < 0:
                    continue
                if amount == 0 and operation_type != "interest":
                    continue
                # В файлах встречаются два формата даты
                try:
                    date = datetime.strptime(
                        row.get("date"), "%Y-%m-%d %H:%M:%S"
                    )
                except ValueError:
                    date = datetime.strptime(
                        row.get("date"), "%d/%m/%Y %H:%M"
                    )
            except (TypeError, ValueError):
                continue

            operation = {
                "type": operation_type,
                "amount": amount,
                "datetime": date,
                "balance": balance_after,
                "status": status,
            }

            if operation not in operations:
                operations.append(operation)

        operations.sort(key=lambda item: item["datetime"])
        return operations

    def load_history(self, file_path: str) -> int:
        """Добавляет очищенную историю из CSV/JSON и обновляет баланс. Возвращает количество добавленных операций"""
        path = Path(file_path)
        with path.open("r", encoding="utf-8", newline="") as file:
            if path.suffix.lower() == ".csv":
                rows = list(csv.DictReader(file))
            elif path.suffix.lower() == ".json":
                rows = json.load(file)
            else:
                raise ValueError("Поддерживаются только CSV и JSON")

        account_rows = [
            row for row in rows
            if row.get("account_number") == self.account_number
        ]
        operations = self.clean_history(account_rows)
        self.operations_history.extend(operations)

        if operations:
            # история уже совершённых операций, так что берём последний остаток
            self._balance = operations[-1]["balance"]

        return len(operations)

In [23]:
account = Account("Эдуард Почуев", 1000)
print("Владелец:", account.holder)
print("Номер счёта:", account.account_number)
print("Пополнение на 500:", account.deposit(500))
print("Снятие 300:", account.withdraw(300))
print("Снятие 2000 при недостатке средств:", account.withdraw(2000))
print("Баланс:", account.get_balance())

for operation in account.get_history():
    print(operation)

Владелец: Эдуард Почуев
Номер счёта: ACC-1000
Пополнение на 500: True
Снятие 300: True
Снятие 2000 при недостатке средств: False
Баланс: 1200.0
{'type': 'deposit', 'amount': 500, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 478611), 'balance': 1500.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 300, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 478643), 'balance': 1200.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 2000, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 478663), 'balance': 1200.0, 'status': 'fail'}


## Задание 2

Расчётный и сберегательный счета.

In [24]:
class CheckingAccount(Account):
    account_type: str = "checking"

class SavingsAccount(Account):
    account_type: str = "savings"
    allowed_operations: tuple[str, ...] = (
        "deposit", "withdraw", "interest"
    )

    def apply_interest(self, rate: float) -> float:
        self._validate_number(rate)
        if rate < 0:
            raise ValueError("Процентная ставка не может быть отрицательной")

        interest = self._balance * rate / 100
        self._balance += interest
        self._add_operation("interest", interest, "success")

        return interest

    def withdraw(self, amount: float) -> bool:
        self._validate_amount(amount)

        if amount > self._balance / 2:
            self._add_operation("withdraw", amount, "fail")
            return False

        return self._withdraw(amount)

In [25]:
checking = CheckingAccount("Эдуард Почуев", 200)
print("Расчётный счёт:", checking.account_number, checking.account_type)
checking.deposit(300)
print("Снятие 500:", checking.withdraw(500))
print("Баланс:", checking.get_balance())

savings = SavingsAccount("Иван Иванов", 1000)
print("\nСберегательный счёт:", savings.account_number, savings.account_type)
savings.deposit(1000)
print("Начислено при ставке 7%:", savings.apply_interest(7))
print("Баланс после процентов:", savings.get_balance())
print("Снятие 1100 — больше половины:", savings.withdraw(1100))
print("Снятие 1070:", savings.withdraw(1070))
print("Баланс:", savings.get_balance())

for operation in savings.get_history():
    print(operation)

Расчётный счёт: ACC-1001 checking
Снятие 500: True
Баланс: 0.0

Сберегательный счёт: ACC-1002 savings
Начислено при ставке 7%: 140.0
Баланс после процентов: 2140.0
Снятие 1100 — больше половины: False
Снятие 1070: True
Баланс: 1070.0
{'type': 'deposit', 'amount': 1000, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 583698), 'balance': 2000.0, 'status': 'success'}
{'type': 'interest', 'amount': 140.0, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 583712), 'balance': 2140.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 1100, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 583753), 'balance': 2140.0, 'status': 'fail'}
{'type': 'withdraw', 'amount': 1070, 'datetime': datetime.datetime(2026, 10, 4, 22, 43, 35, 583773), 'balance': 1070.0, 'status': 'success'}


### Валидация и анализ истории

In [26]:
try:
    Account("иван Иванов")
except ValueError as error:
    print("Некорректное имя:", error)

try:
    account.deposit(-100)
except ValueError as error:
    print("Отрицательная сумма:", error)

print("\nПоследние две операции на сумму от 1000:")
large_operations = savings.show_large_operations(n=2, min_amount=1000)

Некорректное имя: Укажите имя и фамилию с заглавных букв
Отрицательная сумма: Сумма операции должна быть положительной

Последние две операции на сумму от 1000:
04.10.2026 22:43:35 | withdraw | сумма: 1070.00 | баланс: 1070.00
04.10.2026 22:43:35 | deposit | сумма: 1000.00 | баланс: 2000.00


## Задание 3

Загрузка и очистка предоставленных CSV/JSON. Общие методы
`load_history()` и `clean_history()` наследуются от `Account`.

In [27]:
checking_archive = CheckingAccount("Эдуард Почуев")
checking_archive.account_number = "ACC-100001"
loaded_csv = checking_archive.load_history("data/transactions_dirty.csv")

print("CSV, счёт:", checking_archive.account_number)
print("Добавлено операций:", loaded_csv)
print(f"Баланс: {checking_archive.get_balance():.2f}")

for operation in checking_archive.get_history():
    print(operation)

CSV, счёт: ACC-100001
Добавлено операций: 17
Баланс: 7959.00
{'type': 'deposit', 'amount': 921.0, 'datetime': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance': 2121.0, 'status': 'success'}
{'type': 'deposit', 'amount': 607.0, 'datetime': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance': 2728.0, 'status': 'success'}
{'type': 'deposit', 'amount': 129.0, 'datetime': datetime.datetime(2025, 9, 28, 22, 17), 'balance': 3345.0, 'status': 'success'}
{'type': 'deposit', 'amount': 488.0, 'datetime': datetime.datetime(2025, 9, 28, 22, 17, 26), 'balance': 3216.0, 'status': 'success'}
{'type': 'deposit', 'amount': 880.0, 'datetime': datetime.datetime(2025, 9, 29, 22, 17, 26), 'balance': 4225.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 352.0, 'datetime': datetime.datetime(2025, 10, 1, 22, 17, 26), 'balance': 3687.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 65.0, 'datetime': datetime.datetime(2025, 10, 1, 22, 17, 26), 'balance': 3622.0, 'status': 'success'}
{'type': 'w

In [28]:
savings_archive = SavingsAccount("Иван Иванов")
savings_archive.account_number = "ACC-100002"
loaded_json = savings_archive.load_history("data/transactions_dirty.json")

print("JSON, счёт:", savings_archive.account_number)
print("Добавлено операций:", loaded_json)
print(f"Баланс: {savings_archive.get_balance():.2f}")

for operation in savings_archive.get_history():
    print(operation)

JSON, счёт: ACC-100002
Добавлено операций: 18
Баланс: 5605.79
{'type': 'deposit', 'amount': 469.0, 'datetime': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance': 1969.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 392.0, 'datetime': datetime.datetime(2025, 9, 27, 22, 17, 26), 'balance': 1577.0, 'status': 'success'}
{'type': 'deposit', 'amount': 317.0, 'datetime': datetime.datetime(2025, 9, 28, 22, 17, 26), 'balance': 1894.0, 'status': 'success'}
{'type': 'deposit', 'amount': 840.0, 'datetime': datetime.datetime(2025, 10, 2, 22, 17, 26), 'balance': 3665.55, 'status': 'success'}
{'type': 'interest', 'amount': 183.28, 'datetime': datetime.datetime(2025, 10, 4, 22, 17), 'balance': 3848.83, 'status': 'success'}
{'type': 'interest', 'amount': 192.44, 'datetime': datetime.datetime(2025, 10, 6, 22, 17, 26), 'balance': 4041.27, 'status': 'success'}
{'type': 'deposit', 'amount': 426.0, 'datetime': datetime.datetime(2025, 10, 8, 22, 17, 26), 'balance': 4603.27, 'status': 'success'}
{'